<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/File_Carving.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To simulate a basic digital forensic file-carving investigation by scanning binary data for known file signatures, identifying their byte offsets, and extracting recognizable file sections into separate recovered files.

**Algorithm**

1. Prepare Data:
Create a binary file containing unrelated data and recognizable file signatures.

2. Load Signatures:
Maintain a collection of known file headers and footers.

3. Scan Bytes:
Read the binary file and search for known signatures.

4. Record Offset:
Record the exact byte offset where each signature is found.

5. Identify Type:
Determine the file type from the detected signature.

6. Extract Data:
Extract the recognizable section between the header and appropriate footer where possible.

7. Save Recovery:
Save each recovered section as a separate file.

8. Generate Summary:
Display the signature, byte offset, recovered filename, and file type.

In [1]:
import os

# ---------------------------------------------------------
# BASIC FILE-CARVING INVESTIGATION
# ---------------------------------------------------------

print("=" * 70)
print("              DIGITAL FORENSIC FILE CARVING")
print("=" * 70)


# ---------------------------------------------------------
# STEP 1: CREATE SIMULATED BINARY EVIDENCE
# ---------------------------------------------------------

evidence_file = "binary_evidence.bin"

# Simulated PDF data
pdf_data = (
    b"%PDF-1.4\n"
    b"This is a simulated PDF file recovered from binary evidence.\n"
    b"%%EOF"
)

# Simulated JPEG data
jpeg_data = (
    b"\xFF\xD8\xFF"
    b"\xE0\x00\x10JFIF"
    b"SIMULATED JPEG IMAGE DATA"
    b"\xFF\xD9"
)

# Simulated PNG data
png_data = (
    b"\x89PNG\r\n\x1a\n"
    b"SIMULATED PNG IMAGE DATA"
    b"IEND"
)

# Unrelated binary data
unrelated_data = (
    b"\x00\x11\x22\x33\x44\x55"
    b"RANDOM_FORENSIC_DATA"
    b"\x99\x88\x77\x66"
)

# Combine data
binary_content = (
    unrelated_data +
    b"\xAA\xBB\xCC" +
    pdf_data +
    b"\x10\x20\x30\x40" +
    jpeg_data +
    b"\x50\x60\x70" +
    png_data +
    b"\xFF\xEE\xDD"
)

# Write binary evidence
with open(evidence_file, "wb") as file:
    file.write(binary_content)

print("\nSimulated binary evidence created:")
print(evidence_file)

print("Evidence size:",
      os.path.getsize(evidence_file),
      "bytes")


# ---------------------------------------------------------
# STEP 2: DEFINE FILE SIGNATURES
# ---------------------------------------------------------

signatures = {

    "PDF": {
        "header": b"%PDF",
        "footer": b"%%EOF",
        "extension": ".pdf"
    },

    "JPEG": {
        "header": b"\xFF\xD8\xFF",
        "footer": b"\xFF\xD9",
        "extension": ".jpg"
    },

    "PNG": {
        "header": b"\x89PNG\r\n\x1a\n",
        "footer": b"IEND",
        "extension": ".png"
    }
}


# ---------------------------------------------------------
# STEP 3: READ BINARY CONTENT
# ---------------------------------------------------------

with open(evidence_file, "rb") as file:
    data = file.read()


# ---------------------------------------------------------
# STEP 4: SCAN FOR SIGNATURES
# ---------------------------------------------------------

recovered_files = []

print("\n" + "=" * 70)
print("SCANNING BINARY EVIDENCE")
print("=" * 70)

for file_type, signature_info in signatures.items():

    header = signature_info["header"]
    footer = signature_info["footer"]
    extension = signature_info["extension"]

    search_position = 0

    while True:

        offset = data.find(header, search_position)

        if offset == -1:
            break

        print(
            f"\nSignature detected: {file_type}"
        )

        print(
            f"Byte offset: {offset}"
        )

        # -------------------------------------------------
        # FIND FOOTER
        # -------------------------------------------------

        footer_position = data.find(
            footer,
            offset + len(header)
        )

        if footer_position != -1:

            end_position = (
                footer_position + len(footer)
            )

            recovered_data = data[
                offset:end_position
            ]

            recovered_name = (
                f"recovered_{len(recovered_files) + 1}"
                + extension
            )

            # Save recovered section
            with open(
                recovered_name,
                "wb"
            ) as recovered_file:

                recovered_file.write(
                    recovered_data
                )

            recovered_files.append({
                "signature": header.hex(" "),
                "offset": offset,
                "filename": recovered_name,
                "type": file_type,
                "size": len(recovered_data)
            })

            print(
                f"Recovered file: {recovered_name}"
            )

        else:

            print(
                "Footer not found - "
                "section could not be fully recovered."
            )

        search_position = offset + 1


# ---------------------------------------------------------
# STEP 5: RECOVERY SUMMARY
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("FILE RECOVERY SUMMARY")
print("=" * 70)

if recovered_files:

    for number, result in enumerate(
        recovered_files,
        1
    ):

        print(f"\nRecovery {number}")
        print("-" * 40)

        print(
            "Signature :",
            result["signature"]
        )

        print(
            "Offset    :",
            result["offset"],
            "bytes"
        )

        print(
            "File Name :",
            result["filename"]
        )

        print(
            "File Type :",
            result["type"]
        )

        print(
            "File Size :",
            result["size"],
            "bytes"
        )

else:

    print("No recognizable file signatures found.")


# ---------------------------------------------------------
# STEP 6: FINAL RESULT
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("CARVING RESULT")
print("=" * 70)

print(
    "Total recovered files:",
    len(recovered_files)
)

print(
    "Original evidence:",
    evidence_file
)

print("\nFile carving investigation completed.")
print("=" * 70)

              DIGITAL FORENSIC FILE CARVING

Simulated binary evidence created:
binary_evidence.bin
Evidence size: 191 bytes

SCANNING BINARY EVIDENCE

Signature detected: PDF
Byte offset: 33
Recovered file: recovered_1.pdf

Signature detected: JPEG
Byte offset: 112
Recovered file: recovered_2.jpg

Signature detected: PNG
Byte offset: 152
Recovered file: recovered_3.png

FILE RECOVERY SUMMARY

Recovery 1
----------------------------------------
Signature : 25 50 44 46
Offset    : 33 bytes
File Name : recovered_1.pdf
File Type : PDF
File Size : 75 bytes

Recovery 2
----------------------------------------
Signature : ff d8 ff
Offset    : 112 bytes
File Name : recovered_2.jpg
File Type : JPEG
File Size : 37 bytes

Recovery 3
----------------------------------------
Signature : 89 50 4e 47 0d 0a 1a 0a
Offset    : 152 bytes
File Name : recovered_3.png
File Type : PNG
File Size : 36 bytes

CARVING RESULT
Total recovered files: 3
Original evidence: binary_evidence.bin

File carving investiga

**Result**

The Python program successfully simulated a file-carving investigation by scanning binary evidence for recognizable file signatures. It identified PDF, JPEG, and PNG headers, recorded their byte offsets, extracted the recognizable sections, and saved them as separate recovered files. The recovery summary displayed the signature detected, byte location, recovered filename, file type, and recovered size, demonstrating the basic principle of recovering files from raw binary data without relying on filesystem metadata.